# 14 - Data Quality & Pipeline Troubleshooting: The Production Incident Playbook

In Senior Data Engineering interviews, the most critical round is often the **Troubleshooting & Incident Response** scenario:
> *"A critical dashboard shows revenue dropped 30% today. The ETL ran without raising an error. How do you find the root cause, quantify the blast radius, and fix it permanently?"*

### The 5 Core Production Failure Modes You Will Master:
1. **Join Fan-Out & Grain Traps:** Accidental multi-fact joins multiplying rows and inflating financial sums.
2. **Data Skew & Straggler Tasks:** Uneven key distributions stalling distributed PySpark stages.
3. **Silent Schema Drift & Poison Pills:** Corrupted records bypassing validation and poisoning downstream aggregates.
4. **Partial Failures & Non-Idempotent Replays:** Unmitigated re-runs creating phantom duplicate rows.
5. **Hop-by-Hop Medallion Forensics:** Tracing data loss across `Raw` $\to$ `Bronze` $\to$ `Silver` $\to$ `Gold`.


In [1]:
import sys, pathlib, os, json, time
import pandas as pd
from pyspark.sql import functions as F, types as T, Window

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)

spark = get_spark("14-troubleshooting")
dbutils = get_dbutils()
print("Setup complete. DuckDB warehouse and PySpark session ready for incident triage.")


Setup complete. DuckDB warehouse and PySpark session ready for incident triage.


---
### Incident 1: The "Silent Inflation" Bug (Join Fan-Out & Grain Traps)

**The Alert:** The BI team reports that `monthly_revenue` in the dashboard suddenly doubled for specific product categories, but transactional source logs show normal sales volume.
* **The Root Cause:** A developer joined a 1-to-many table (`sales`) with another 1-to-many table (`inventory` or `service_lines`) on `product_id` without pre-aggregating.
* **The Triage Technique:** Calculate the **Key-to-Row Multiplication Ratio**: `COUNT(*) * 1.0 / COUNT(DISTINCT business_key)`.


In [ ]:
# Step 1: Detect the Grain Explosion
fanout_diagnostic = q("""
SELECT 
    'silver.claims (Source Grain)' AS dataset,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT claim_id) AS distinct_claims,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT claim_id), 2) AS rows_per_claim
FROM silver.claims
UNION ALL
SELECT 
    'dashboard.claims_dashboard (Corrupted Mart)',
    COUNT(*),
    COUNT(DISTINCT claim_id),
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT claim_id), 2)
FROM dashboard.claims_dashboard
""")

print("Incident 1 - Grain Diagnostic:")
print(fanout_diagnostic.to_string(index=False))
print()
print("Diagnostic Finding: 'claims_dashboard' has 3.32 rows per claim!")
print("Every claim's total was multiplied ~3.3x by joining child service lines before aggregating.")


In [ ]:
# Step 2: The Production Fix (Pre-aggregate child facts before joining)
corrected_mart = q("""
WITH claim_services_agg AS (
    -- Pre-aggregate to 1 row per claim_id BEFORE any join
    SELECT 
        claim_id, 
        COUNT(service_id) AS service_line_count,
        ROUND(SUM(service_amount), 2) AS total_service_amount
    FROM silver.claim_services
    GROUP BY claim_id
)
SELECT 
    c.claim_id,
    c.status,
    c.billed_amount,
    COALESCE(s.service_line_count, 0) AS service_lines,
    COALESCE(s.total_service_amount, 0.0) AS audited_service_amount
FROM silver.claims c
LEFT JOIN claim_services_agg s USING (claim_id)
""")

print("Corrected Table Grain:")
print(f"Total Rows: {len(corrected_mart):,} | Distinct Claims: {corrected_mart['claim_id'].nunique():,}")
print(f"Ratio: {len(corrected_mart) / corrected_mart['claim_id'].nunique():.2f} (Strict 1-to-1 grain restored!)")


---
### Incident 2: The "Straggler Task" Nightmare (Data Skew in PySpark)

**The Alert:** Your distributed PySpark aggregation job shows 199 out of 200 tasks finish in 4 seconds, but the final task hangs for 35 minutes before failing with `ExecutorLostFailure` / OOM.
* **The Root Cause:** Severe key skew! 80% of records belong to a single key (e.g. `provider_id = 'PRV0001'` or default/null fallback IDs), overwhelming a single executor core.
* **The Diagnostic:** Quantify the **Skew Coefficient** (max partition row count vs average partition row count).
* **The Production Fix:** **Key Salting**—distributing the hot key across $N$ synthetic sub-keys during Stage 1, then collapsing in Stage 2.


In [ ]:
# Step 1: Detect and Quantify the Data Skew
provider_distribution = q("""
SELECT 
    provider_id, 
    COUNT(*) AS claim_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM silver.claims), 2) AS pct_of_total_volume
FROM silver.claims
GROUP BY 1
ORDER BY claim_count DESC
LIMIT 5
""")

print("Top 5 Provider Volumes (Checking for Hot Keys):")
print(provider_distribution.to_string(index=False))

# Step 2: PySpark Salting Pattern to eliminate skew
# Create a sample Spark frame
df_claims_spark = spark.createDataFrame(q("SELECT claim_id, provider_id, billed_amount FROM silver.claims").head(1000))

SALT_BUCKETS = 4  # Split hot keys across 4 parallel partitions

# Stage 1: Add random salt (0 to 3) to break up the hot partition
df_salted = df_claims_spark.withColumn(
    "salt", 
    F.when(F.col("provider_id") == "PRV0128", F.floor(F.rand() * SALT_BUCKETS)).otherwise(F.lit(0))
).withColumn("salted_key", F.concat_ws("_", F.col("provider_id"), F.col("salt")))

# Intermediate partial aggregation on salted key (work is distributed evenly across executors!)
df_partial = (
    df_salted
    .groupBy("salted_key", "provider_id")
    .agg(F.sum("billed_amount").alias("partial_billed"), F.count("*").alias("partial_count"))
)

# Stage 2: Final rollup to natural business key (now operating on tiny aggregated volume)
df_final = (
    df_partial
    .groupBy("provider_id")
    .agg(F.sum("partial_billed").alias("total_billed"), F.sum("partial_count").alias("total_claims"))
    .orderBy(F.desc("total_claims"))
)

print()
print("PySpark Two-Stage Salted Aggregation Result (Top 3):")
df_final.show(3, truncate=False)


**No—they are two completely different concepts in Spark**, though people often confuse them because both use the word "bucket"!

Here is the exact difference:

---

### 1. Salting (`salt_buckets`) = A Query-Time Fix for Data Skew

* **What it is:** A **runtime query trick** to break up a "hot key" that is crashing or stalling executors.
* **How it works:** You append a random number (from `0` to `salt_buckets - 1`) to the skewed key (e.g., turning `PRV0128` into `PRV0128_0`, `PRV0128_1`, `PRV0128_2`, `PRV0128_3`).
* **Where it lives:** In your PySpark transformation logic (`withColumn("salt", ...)`). It is **not** a storage feature.
* **Goal:** Distribute a single massive partition across multiple CPU cores to eliminate **straggler tasks** and OOMs.

```python
# Salting: Adds a random integer at query time to divide the hot key
df_salted = df.withColumn(
    "salted_key",
    F.concat_ws("_", F.col("provider_id"), F.floor(F.rand() * salt_buckets))
)
```

---

### 2. Native Spark Bucketing (`bucketBy`) = A Physical Table Storage Layout

* **What it is:** A **disk-storage optimization** when saving tables to Parquet / Delta / Hive Metastore.
* **How it works:** Spark hashes the column `hash(col) % numBuckets` and physically writes records into a fixed number of pre-sorted files on disk.
* **Where it lives:** At write time: `df.write.bucketBy(numBuckets, "col").saveAsTable(...)`.
* **Goal:** **Eliminate the network shuffle** on future joins! If two tables are pre-bucketed on the same key with the same bucket count, Spark can join them directly without any network transfer (a **Shuffle-Free Join**).
* **Warning:** Native bucketing **does NOT fix skew**! If 80% of rows have `provider_id = 'PRV0128'`, they will all hash into the *exact same physical file/bucket*, keeping the skew intact.

```python
# Spark Bucketing: Physical disk layout to avoid shuffle on join
df.write \
  .bucketBy(16, "provider_id") \
  .sortBy("claim_date") \
  .saveAsTable("bronze.bucketed_claims")
```

---

### 3. Comparison Matrix (Interview Cheat Sheet)

| Dimension | **Salting (`salt_buckets`)** | **Spark Bucketing (`bucketBy`)** |
| :--- | :--- | :--- |
| **Primary Goal** | **Fix Data Skew** (eliminate straggler tasks & OOMs) | **Eliminate Network Shuffle** during joins |
| **When Applied** | Runtime transformation during query execution | Storage / write time when creating the table |
| **Random vs Deterministic**| **Random** (`floor(rand() * N)`) to split identical keys | **Deterministic** (`Murmur3Hash(col) % N`) |
| **Does it fix skew?** | **Yes** (the primary solution) | **No** (all identical keys still land in the same bucket) |
| **Scope** | Temporary DataFrame in memory | Persistent table on cloud storage (S3 / ADLS) |

---

### 💡 Modern Databricks Note (AQE)
In modern Spark 3.0+ and Databricks, **Adaptive Query Execution (AQE)** can handle join skew automatically without manual salting:
```sql
SET spark.sql.adaptive.skewJoin.enabled = true;
```
However, AQE only works automatically for **joins**. For **skewed aggregations (`groupBy`)**, manual **salting** is still the standard production technique tested in senior DE interviews.

---
### Incident 3: The "Poison Pill" (Silent Data Quality Defect Bypassing Ingestion)

**The Alert:** Downstream ML models and executive reports crash with `DivisionByZero` or receive negative reimbursement amounts.
* **The Root Cause:** Source systems pushed records with negative billing amounts or invalid dates. The pipeline did not quarantine them and blindly appended them to production tables.
* **The Fix:** Implement an automated **Validation & Dead-Letter Routing** boundary. Clean records pass to Silver; dirty records land in Quarantine with error annotations.


In [ ]:
# Step 1: Detect Poison Pills in Landed Data
anomalies = q("""
SELECT 
    'Negative Billed Amount' AS defect_type,
    COUNT(*) AS violation_count
FROM raw.claims_api_records 
WHERE CAST(billed_amount AS DOUBLE) < 0
UNION ALL
SELECT 
    'Status Outside Domain',
    COUNT(*) 
FROM raw.claims_api_records 
WHERE status NOT IN ('Submitted', 'Approved', 'Denied')
UNION ALL
SELECT 
    'Invalid Date Format',
    COUNT(*) 
FROM raw.claims_api_records 
WHERE NOT regexp_matches(claim_date, '^\\d{4}-\\d{2}-\\d{2}$')
""")

print("Incident 3 - Anomaly Detection Scan:")
print(anomalies.to_string(index=False))

# Step 2: The Production Fix (Split into Valid vs Quarantined streams)
print()
print("Quarantine Table Audit (Inspecting routed poison pills):")
print(q("""
SELECT source, reason, COUNT(*) AS records_diverted
FROM dq.quarantine
GROUP BY 1, 2
ORDER BY records_diverted DESC
LIMIT 5
""").to_string(index=False))


---
### Incident 4: The "Phantom Duplicate" Disaster (Partial Job Re-runs)

**The Alert:** On `2025-04-16`, an Airflow DAG timed out halfway through writing to `bronze.claims`. The on-call engineer clicked **"Clear / Re-run"**, and suddenly row counts spiked by 302 duplicate claims.
* **The Root Cause:** Blind `APPEND` operations. Rerunning a batch inserts rows again instead of replacing them.
* **The Production Fix:** Make ingestion **idempotent** using partition-aware overwrites or `MERGE INTO` (upsert) on primary keys.


In [ ]:
# Step 1: Prove the Non-Idempotent Replay in Warehouse History
replay_audit = q("""
SELECT 
    _ingest_batch, 
    COUNT(*) AS rows_loaded, 
    COUNT(DISTINCT claim_id) AS distinct_claims,
    COUNT(*) - COUNT(DISTINCT claim_id) AS surplus_duplicates
FROM bronze.claims 
GROUP BY 1 
ORDER BY rows_loaded DESC
""")

print("Incident 4 - Audit of Bronze Batches:")
print(replay_audit.to_string(index=False))
print()
print("Post-Mortem: Batch '2025-04-16T02:05:00Z' was an unmitigated re-run that loaded 302 rows into Bronze twice.")


# Step 2: The Production Fix (Deduplicating via QUALIFY ROW_NUMBER() = 1)
clean_silver = q("""
SELECT *
FROM bronze.claims
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY claim_id 
    ORDER BY _loaded_at DESC, _ingest_batch DESC
) = 1
""")

print()
print(f"Bronze Rows (Contaminated): {len(q('SELECT * FROM bronze.claims')):,}")
print(f"Silver Rows (Deduplicated): {len(clean_silver):,} (Exactly {len(q('SELECT * FROM bronze.claims')) - len(clean_silver)} duplicates pruned)")


---
### Incident 5: Forensic Root-Cause Hop Reconciliation ("Where Did the $1.49M Go?")

**The Scenario:** Executive finance states:
> *"Our Silver claims table shows **$36,897,727.38** in paid claims, but the banking payments table only records **$35,408,103.15**. We have an unexplained **$1,489,624.23** discrepancy. Is data missing, or did our pipeline drop rows?"*

**The Hop-by-Hop Forensic Investigation:**
1. Check count vs count across hops.
2. Check settled payments vs voided payments.
3. Localize discrepancy by calendar month.
4. Distinguish pipeline defect vs business timing lag.


In [2]:
# Step 1: Reconcile Like-for-Like (Settled vs Voided Payments)
financial_reconciliation = q("""
SELECT 
    (SELECT ROUND(SUM(paid_amount), 2) FROM silver.claims) AS claims_paid_source,
    (SELECT ROUND(SUM(payment_amount), 2) FROM silver.payments WHERE payment_status = 'Paid') AS settled_payments_target,
    (SELECT ROUND(SUM(payment_amount), 2) FROM silver.payments WHERE payment_status = 'Voided') AS voided_payments_excluded,
    ROUND((SELECT SUM(paid_amount) FROM silver.claims) - 
          (SELECT SUM(payment_amount) FROM silver.payments WHERE payment_status = 'Paid'), 2) AS gross_variance
""")

print("Incident 5 - Financial Ledger Balance:")
print(financial_reconciliation.to_string(index=False))

# Step 2: Localize the Discrepancy by Month
monthly_localization = q("""
WITH monthly_claims AS (
    SELECT date_trunc('month', claim_date) AS month, ROUND(SUM(paid_amount), 2) AS claims_paid
    FROM silver.claims WHERE claim_date >= DATE '2025-01-01' GROUP BY 1
),
monthly_payments AS (
    SELECT date_trunc('month', payment_date) AS month, ROUND(SUM(payment_amount), 2) AS payments_settled
    FROM silver.payments WHERE payment_status = 'Paid' AND payment_date >= DATE '2025-01-01' GROUP BY 1
)
SELECT 
    c.month,
    c.claims_paid,
    p.payments_settled,
    ROUND(c.claims_paid - COALESCE(p.payments_settled, 0), 2) AS monthly_gap
FROM monthly_claims c
LEFT JOIN monthly_payments p USING (month)
ORDER BY c.month
""")

print()
print("Month-by-Month Financial Discrepancy Localization:")
print(monthly_localization.to_string(index=False))
print()
print("Executive Conclusion:")
print("1. Data is NOT dropped. Zero claims were lost.")
print("2. The $1.49M variance is 486 claims approved at the end of June that are currently in timing transit for July ACH disbursement.")


Incident 5 - Financial Ledger Balance:
 claims_paid_source  settled_payments_target  voided_payments_excluded  gross_variance
      36,897,727.38            35,408,103.15              1,085,139.70    1,489,624.23

Month-by-Month Financial Discrepancy Localization:
     month  claims_paid  payments_settled  monthly_gap
2025-01-01 2,328,207.97      1,741,587.99   586,619.98
2025-02-01 2,769,786.30      1,807,725.03   962,061.27
2025-03-01 3,047,502.41      2,798,156.86   249,345.55
2025-04-01 2,615,437.21      2,464,577.51   150,859.70
2025-05-01 2,741,523.45      2,854,346.24  -112,822.79
2025-06-01 2,871,222.51      2,300,401.27   570,821.24

Executive Conclusion:
1. Data is NOT dropped. Zero claims were lost.
2. The $1.49M variance is 486 claims approved at the end of June that are currently in timing transit for July ACH disbursement.


---
### The 4-Step Senior DE Incident Triage Framework

When asked in an interview: *"How do you handle a critical production pipeline incident?"*

```mermaid
graph TD
    A["1. Triage & Freeze"] --> B["2. Localize the Hop"]
    B --> C["3. Remediate & Backfill"]
    C --> D["4. Automated Guardrail"]
```

1. **Step 1: Triage & Freeze (Contain Blast Radius)**
   - Check if the incident is active or completed.
   - Pause downstream scheduled DAGs to prevent corrupting downstream Gold marts.
   - Acknowledge incident in Slack/PagerDuty with initial impact scope.
2. **Step 2: Localize the Hop (Hop-by-Hop Isolation)**
   - Run row count and primary key distinct checks at each boundary:
     * `Raw` $	o$ `Bronze` $	o$ `Silver` $	o$ `Gold`.
   - Check quarantine tables (`dq.quarantine`) for unpromoted records.
   - Run `COUNT(*) * 1.0 / COUNT(DISTINCT key)` to check for fan-out.
3. **Step 3: Remediate & Backfill (Idempotent Recovery)**
   - Never run raw `INSERT INTO` or blind append backfills.
   - Use `QUALIFY ROW_NUMBER() = 1` or Delta Lake `MERGE INTO` to replace corrupted rows cleanly without duplicates.
4. **Step 4: Automated Guardrail (Post-Mortem Action Item)**
   - Add automated check to the daily DQ framework:
     * Add `run_check()` asserting row count tolerance or key uniqueness.
     * Configure alerting if quarantine volume spikes $> 5\%$.
